[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Modern_Architectures.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Modern Architectures

The transformer's descendants, dissected: mixture-of-experts routing (capacity without compute), attention's cost curve and its linear/sliding-window repairs, and where [SSM blocks](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) fit. Each mechanism built small and measured.

## 1. Pre-requisites

[Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb), [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb), [Scale_NN](./Scale_NN/Scale_NN.ipynb).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
torch.manual_seed(0)

---
### 🕐 Session 1 of 3 — *Attention's Cost Curve* (~35 min)
**Goal:** measure the quadratic wall; see what sliding windows and linear attention trade away.
**Builds on:** [Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (MoE).

---

## 2. The Quadratic Wall, Measured

💡 **Intuition.** Full attention lets every token query every token: $O(T^2)$ compute and memory — the price of unlimited connectivity. The repairs each *remove* something: **sliding windows** keep only local links (recover long range by stacking layers — the [CNN receptive-field](./Intro_CNN/Intro_CNN.ipynb) trick); **linear attention** replaces softmax with a kernel so the sum factorizes into a running state ($O(T)$ — and mathematically an [SSM](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb)!). No free lunch: each buys speed with a connectivity prior.

In [ ]:

# YOUR CODE HERE


**What just happened.** Three attention variants timed at three sequence lengths:

| $T$ | full | window | linear |
|---|---|---|---|
| 512 | 3.1 ms | 0.4 ms | 0.7 ms |
| 2048 | 4.5 ms | 0.8 ms | 0.8 ms |
| 8192 | **68.6 ms** | 3.2 ms | 1.8 ms |

**Check the printed claim against the table, because it is only half supported.** "~16× per 4× length" holds for the second step: $4.5 \to 68.6$ ms is a factor of **15.2**, textbook quadratic. It does **not** hold for the first: $3.1 \to 4.5$ ms is a factor of **1.45**, nowhere near 16. The quadratic wall is visible in one of the two steps, and the discrepancy is the most instructive thing in the cell.

**The explanation is the overhead-bound regime from [Scale_NN](./Scale_NN/Scale_NN.ipynb) Session 2.** At $T = 512$ the $512\times512$ matmul is a few hundred microseconds of real arithmetic; the other ~2.8 ms is Python dispatch, tensor allocation, and BLAS setup — **fixed costs independent of $T$**. Only at $T = 8192$, where the score matrix has 67 million entries, does arithmetic dominate and the asymptotics emerge. **A benchmark that has not escaped its own overhead is measuring the framework, not the algorithm.**

**So the right reading is: the quadratic scaling is real and is confirmed by the 2048 → 8192 step alone.** Put the consequence in memory terms, where it bites hardest: at $T = 100{,}000$ the score matrix is $10^{10}$ entries — **40 GB in fp32, for one head in one layer**. That is why long context was an unsolved engineering problem rather than a matter of waiting for faster chips.

**Note the ordering upset at $T = 8192$, since it is not what the theory predicts.** Linear attention (1.8 ms) beats the sliding window (3.2 ms), despite the window doing less total arithmetic. The reason is in the code: `window_attn` is a **Python loop** over 128 blocks, paying dispatch overhead 128 times, while `linear_attn` is three fused matrix operations. A proper fused windowed kernel — which is what FlashAttention's windowed variants provide — would reverse this. **Asymptotic cost and implementation quality are different axes, and a timing table conflates them.**

**Two caveats on the measurement itself.** These are **single unwarmed runs** with no repeats and no error bars, so treat each number as good to perhaps ±50%; the [Scale_NN](./Scale_NN/Scale_NN.ipynb) throughput cell shows the warmup discipline this one skips. And all three are forward-pass only at one head and $d = 64$ — real cost is dominated by the backward pass and by many heads across many layers.

**Finally, keep the trade-off in view, because speed is not the whole story.** Full attention can put nearly all its weight on one token 8,000 positions back. A sliding window **cannot see that token at all** in a single layer, and recovers reach only by stacking. Linear attention compresses the entire past into a fixed $d\times d$ state, so it can never attend sharply to one distant token — the summary has already blurred it. **Each speedup is a deletion of connectivity**, and the right question about any efficient-attention paper is which connections it decided not to need.

---
### 🕐 Session 2 of 3 — *Mixture of Experts* (~40 min)
**Goal:** route tokens to specialists: parameters without proportional compute — specialization measured.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (the assembled zoo).

---

## 3. Capacity Without the Bill

💡 **Intuition.** An MoE layer holds $E$ expert MLPs but a learned **router** sends each token to only the top-$k$ — so parameters scale with $E$ while per-token compute scales with $k$. The bet: tokens differ in *kind*, and specialists beat one generalist of equal compute. The classic failure is **routing collapse** (all tokens to one expert), patched with load-balancing losses. We build a 4-expert layer on a task with planted sub-populations and *check who goes where*.

In [ ]:
# task with 4 planted regimes: y depends on x differently per quadrant of a latent code
        # load-balancing auxiliary: encourage uniform expert usage
# THE FAILURE, on purpose: no balancing, no exploration → routing collapse
# THE CURE: load-balancing loss + routing noise

# YOUR CODE HERE


**What just happened.** The same 4-expert layer trained twice, and the difference is not subtle:

| | usage | purity | MSE |
|---|---|---|---|
| no balancing | `[3153, 0, 11, 2836]` | `[0.48, nan, 0.64, 0.52]` | 0.0041 |
| balanced + noisy | `[1513, 1499, 1542, 1446]` | `[0.81, 0.51, 0.74, 0.51]` | **0.0016** |

**Read the first row as the failure it is: one expert received zero tokens and another eleven.** Half the layer's parameters were never trained and are never used. **You paid for a 4-expert model and got a 2-expert model** — and with it, every bit of the parameter-efficiency arithmetic that motivates MoE in the first place.

**The collapse is self-reinforcing, which is why it needs an explicit cure.** An expert that happens to receive more tokens early trains faster, becomes better, and is therefore routed *more* tokens. The rich get richer; the poor get no gradient at all and never recover. **Routing collapse is not a rare pathology — it is the default outcome**, and every production MoE ships with machinery to prevent it.

**The two interventions attack different halves of that loop.** The **load-balancing auxiliary** penalises non-uniform usage directly, and the **routing noise** provides exploration so a currently-worse expert still receives occasional tokens and gets a chance to improve. Penalty plus exploration; real systems add a third, a hard capacity limit that drops tokens beyond each expert's quota.

**The payoff is measured, not asserted: MSE 0.0041 → 0.0016, a 2.6× improvement.** Both models have **identical parameter counts**. The entire difference is whether the capacity gets used, which is a good demonstration that in MoE the routing is the model.

**Now the column that only partly worked, because glossing over it would be dishonest.** Purity is the fraction of an expert's tokens coming from its most common planted regime; random assignment over 4 regimes gives about **0.26**. Two experts hit 0.81 and 0.74 — clear specialisation. **The other two sit at 0.51, meaning each is serving a blend of regimes.** So the correct summary is: balancing forced *usage* to be uniform, and specialisation followed **partially**.

**There is a real tension underneath that, worth naming.** The auxiliary loss rewards uniform *assignment*, which is not the same as correct *specialisation*. If the planted regimes were unevenly sized, a perfectly specialised router would produce **unbalanced** usage — and the balancing loss would penalise it. Turn the penalty up far enough and you get four experts sharing every regime equally: perfectly balanced, perfectly useless. **The balancing weight is a dial between collapse and homogenisation**, and 0.1 here happens to sit in a workable spot.

**One caveat before generalising from this demo.** The regimes were planted, equally sized, and linearly encoded in the first two input coordinates — close to the easiest routing problem that could exist. Real token distributions are not balanced, and experts in real MoE models specialise far less cleanly than the word "expert" suggests; interpretability work generally finds them capturing shallow token-level patterns rather than semantic domains. **The mechanism is real, the tidy specialisation is a property of this dataset.**

---
### 🕐 Session 3 of 3 — *The Assembled Zoo* (~30 min)
**Goal:** how the pieces combine in 2026-era models; the design-space map.
**Builds on:** Session 2.

---

## 4. The Map

| Need | Mechanism | Cost model |
|---|---|---|
| Unlimited connectivity | full attention | $O(T^2)$, KV cache $O(T)$ |
| Long context, cheap | sliding window + a few global layers | $O(Tw)$ |
| Constant-state streaming | linear attention / [SSM/Mamba](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) | $O(T)$, state $O(1)$ |
| Parameters ≫ compute | MoE (top-k routing) | params ×E, FLOPs ×k |
| Memory during training | grouped/multi-query attention, [gradient accumulation](./Scale_NN/Scale_NN.ipynb) | smaller KV, same math |

💡 **Intuition.** Modern frontier models are *hybrids by necessity*: interleaved sliding/full attention, MoE feed-forwards, sometimes SSM layers — each mechanism spending a different currency (compute, memory, connectivity). Read any architecture paper as a walk through this table.

**Exercise with teeth:** wire the MoE layer into the [nano-GPT](./LLMs_from_the_Ground_Up.ipynb) and measure perplexity per FLOP against the dense baseline.

---
## Where next

- [State-Space Models](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) — the third pillar, in depth.
- [Scale_NN](./Scale_NN/Scale_NN.ipynb) — why these trade-offs exist at all.